# Session 22: Threads and processes in Python

In Exercise 1 of Lecture 14 your batch job had 4 CPUs and kept only one of them busy. A Python program runs one
instruction at a time, so the other three CPUs had nothing to do. To use them, a program needs several workers, and
Python gives us two kinds: threads and processes.

In this notebook we start both kinds by hand, we look at what they share, and we measure what each one is good for.

- **Part 1 (Exercise 1)**: threads, here in your JupyterLab session.
- **Part 2 (Exercise 2)**: processes, here in your session and then in batch jobs on the cluster.
- **Part 3 (Exercise 3)**: races and locks.
- **Part 4**: homework.

Run the cells in order, from top to bottom: click a cell and press Shift+Enter, on Mac and on Windows. When the text
says **Predict**, decide on your answer before you run the next cell.

## Before you start

Look at the top right corner of this notebook. It must say **OSPC (Python 3.12)**. If it says something else, click
on that name and choose OSPC (Python 3.12) from the list. If it is not in the list, you have not registered the
kernel yet: run the command of slide 6 of Lecture 14 in a terminal (File > New > Terminal), reload the page, and
choose the kernel.

The next cell imports what we need and prints a few facts about this Python process. The first line must start
with 3.12.

In [ ]:
import multiprocessing
import os
import sys
import threading
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

print("Python:", sys.version.split()[0])
print("Number of this process (PID):", os.getpid())
print("CPUs of the whole node:", os.cpu_count())
print("CPUs of your session:", len(os.sched_getaffinity(0)))

`os.cpu_count()` counts all the CPUs of the node. `os.sched_getaffinity(0)` is the set of CPUs Slurm gave to you.
Your JupyterLab session has 2 CPUs, and on our nodes two CPUs are the two hardware threads of one core. So this
notebook runs on **one core**. Keep that in mind in Part 2.

**WARNING**: never run `ProcessPoolExecutor()` or `multiprocessing.Pool()` without a number on the cluster. Python
would use the 512 CPUs of the node and try to start 512 workers. Your session would crash or hang, and you would have
to restart the kernel (Kernel > Restart Kernel...). Always write the number of workers.

## Part 1. Threads

### Start and join

A process starts with one thread, the **main thread**. Here it is the thread that runs your cells. `threading.Thread`
creates one more thread: `target` is the function it will run, and `args` is the tuple of arguments for that function.

`start()` starts the thread and returns at once, so the main thread goes on while the new thread runs. `join()` makes
the main thread wait until that thread has finished.

**Predict**: will the four threads print the same process number? The same thread number?

In [ ]:
def hello(name):
    # os.getpid() is the number of the process, threading.get_native_id() the number of the thread.
    print(f"{name}: process {os.getpid()}, thread {threading.get_native_id()}")

# Create one Thread object per worker. Nothing runs yet.
workers = [threading.Thread(target=hello, args=(name,)) for name in ["A", "B", "C", "D"]]

for t in workers:
    t.start()   # The thread starts to run hello. start() returns at once.
for t in workers:
    t.join()    # Wait here until this thread has finished.

hello("main")

### Threads share memory

All the threads of a process see the same variables. A thread cannot hand a `return` value back to the main thread,
so we give each thread a place in a shared list and let it write its result there. There is one list, and no copies.

In [ ]:
results = [None] * 4        # One list, seen by every thread.

def square(i):
    results[i] = i * i      # Each thread writes to its own position.

workers = [threading.Thread(target=square, args=(i,)) for i in range(4)]
for t in workers:
    t.start()
for t in workers:
    t.join()

print(results)

### Waiting together

Many programs spend their time waiting: for a web server, for a disk, for a database. `time.sleep(1)` stands for one
such wait. First we "download" four files, one after the other.

In [ ]:
def download(i):
    time.sleep(1)                   # Stands for one second of waiting for a web server.
    results[i] = f"file {i}"

start = time.perf_counter()
for i in range(4):
    download(i)
print(f"One after the other: {time.perf_counter() - start:.2f} s")

**Predict**: how long will the same four downloads take with four threads, on your one core?

In [ ]:
start = time.perf_counter()

workers = [threading.Thread(target=download, args=(i,)) for i in range(4)]
for t in workers:
    t.start()
for t in workers:
    t.join()

print(f"4 threads: {time.perf_counter() - start:.2f} s")
print(results)

Stop here and answer the questions of **Exercise 1** on the slides.

## Part 2. Processes

### A process gets its own copy of the memory

`multiprocessing.Process` works like `threading.Thread`: `target`, `args`, `start()`, `join()`. The difference is
what it creates. On the cluster, Python asks the operating system to `fork`: the new process starts with a copy of
the memory of the notebook's process, and from then on the two are separate.

**Predict**: `change` sets `data[0]` to 1 in the new process. What will `data` be in the notebook afterwards?

In [ ]:
data = [0, 0, 0, 0]

def change(i):
    data[i] = 1
    print(f"inside process {os.getpid()}: data = {data}")

p = multiprocessing.Process(target=change, args=(0,))
p.start()
p.join()

print(f"in the notebook, process {os.getpid()}: data = {data}")

The same function in a thread, for comparison. Threads and the notebook share `data`.

In [ ]:
t = threading.Thread(target=change, args=(0,))
t.start()
t.join()

print(f"in the notebook, process {os.getpid()}: data = {data}")

### A pool of workers

Starting workers by hand and collecting their results in a list is tedious. `concurrent.futures` does it for us.
`ThreadPoolExecutor(4)` starts 4 threads and `ProcessPoolExecutor(4)` starts 4 processes. `ex.map(f, items)` gives each
free worker one item, and returns the results in the order of the items. For a process pool, each item and each
result is copied between the processes. At the end of the `with` block the workers stop.

Now the work is computing, not waiting. `work` is a Python loop, and `tasks` is a list of four of them, 10 million
steps each.

**Predict**: here in your session, on one core, how do the three times below compare?

In [ ]:
def work(n):
    total = 0
    for i in range(n):
        total += i
    return total

tasks = [10_000_000] * 4

start = time.perf_counter()
results = list(map(work, tasks))
print(f"One after the other: {time.perf_counter() - start:.2f} s")

start = time.perf_counter()
with ThreadPoolExecutor(4) as ex:
    results = list(ex.map(work, tasks))
print(f"4 threads: {time.perf_counter() - start:.2f} s")

start = time.perf_counter()
with ProcessPoolExecutor(4) as ex:
    results = list(ex.map(work, tasks))
print(f"4 processes: {time.perf_counter() - start:.2f} s")

One core can only do so much, so we move to a batch job with more CPUs. As in Exercise 1 of Lecture 14, we submit
it with `sbatch`, but from this notebook. The next cell writes the program to a file, `compare.py`, next to this notebook.
It runs the same four tasks in four ways, all in one job, so all four lines are measured on the same node. Each way
runs three times and we keep the shortest time, as `timeit` does, because other work on the node can only slow us down.

In [ ]:
%%writefile compare.py
import os
import time
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

TASKS = [10_000_000] * 4


def work(n):
    total = 0
    for i in range(n):
        total += i
    return total


def one_after_the_other():
    return list(map(work, TASKS))


def in_a_pool(Pool, workers):
    with Pool(workers) as ex:
        return list(ex.map(work, TASKS))


def best_of_3(function, *args):
    times = []
    for _ in range(3):
        start = time.perf_counter()
        function(*args)
        times.append(time.perf_counter() - start)
    return min(times)


if __name__ == "__main__":
    print(f"Node {os.uname().nodename}, {len(os.sched_getaffinity(0))} CPUs")
    print(f"One after the other: {best_of_3(one_after_the_other):.2f} s")
    print(f"4 threads:           {best_of_3(in_a_pool, ThreadPoolExecutor, 4):.2f} s")
    print(f"2 processes:         {best_of_3(in_a_pool, ProcessPoolExecutor, 2):.2f} s")
    print(f"4 processes:         {best_of_3(in_a_pool, ProcessPoolExecutor, 4):.2f} s")

A batch job gets its programs from `PATH`. The next line puts the course environment first, so that `python` in the
job is the Python 3.12 of this kernel. The line only changes this notebook, and you need to run it once.

In [ ]:
os.environ["PATH"] = "/data/software/envs/ospc/bin:" + os.environ["PATH"]

In a notebook, a line that starts with `!` runs in the shell. The options are the ones of Lecture 14: partition `cpu`,
4 CPUs, 2 GB of memory, a limit of 2 minutes. `-o` names the output file. `--wait` keeps the cell running until the job
has finished: you see `[*]` next to the cell, and the whole class shares the cluster, so it can take up to about half a
minute.

**Predict**: with 4 CPUs, how much faster than one after the other will 4 threads be? 2 processes? 4 processes?

In [ ]:
!sbatch -p cpu -c 4 --mem=2G -t 2 -o compare-4cpus.out --wait --wrap "python compare.py"
!cat compare-4cpus.out

**Predict**: with 8 CPUs, which lines will change?

In [ ]:
!sbatch -p cpu -c 8 --mem=2G -t 2 -o compare-8cpus.out --wait --wrap "python compare.py"
!cat compare-8cpus.out

Your job can land on haskell, julia or fortran, and the nodes are not equally fast, so your times can differ from your
neighbor's. Inside one job, all the lines come from the same node.

`sacct` lists your jobs of today, with the CPUs they had and the CPU time they used.

In [ ]:
!sacct -o JobID,JobName%12,AllocCPUS,Elapsed,TotalCPU,State | tail -4

Stop here and answer the questions of **Exercise 2** on the slides.

## Part 3. Races and locks

### Threads: a wait between read and write

`deposit` adds 1 to a shared balance, in three steps: read the balance, wait a moment, write the new value. The wait
stands for asking a database or a file. `time.sleep(0)` is the shortest wait there is: the thread lets the others
run, then comes back.

**Predict**: four threads make 10,000 deposits each. What will the balance be? Run the cell a few times.

In [ ]:
balance = 0

def deposit(n):
    global balance                  # balance is the variable shared by all the threads.
    for _ in range(n):
        current = balance           # Read.
        time.sleep(0)               # Wait.
        balance = current + 1       # Write.

start = time.perf_counter()
workers = [threading.Thread(target=deposit, args=(10_000,)) for _ in range(4)]
for t in workers:
    t.start()
for t in workers:
    t.join()

print(f"Balance: {balance:,} of 40,000, in {time.perf_counter() - start:.2f} s")

A **lock** lets one thread at a time through a piece of code. `with lock:` takes the lock, or waits for it if another
thread holds it, and lets it go at the end of the block. The three steps now form a critical region.

In [ ]:
balance = 0
lock = threading.Lock()

def safe_deposit(n):
    global balance
    for _ in range(n):
        with lock:
            current = balance
            time.sleep(0)
            balance = current + 1

start = time.perf_counter()
workers = [threading.Thread(target=safe_deposit, args=(10_000,)) for _ in range(4)]
for t in workers:
    t.start()
for t in workers:
    t.join()

print(f"Balance: {balance:,} of 40,000, in {time.perf_counter() - start:.2f} s")

### Processes: a counter in shared memory

Processes do not share memory unless we ask for it. `multiprocessing.Value("i", 0)` puts one integer in memory that
all the processes can reach. `counter.value += 1` is again three steps: read, add 1, write. There is no wait this time,
but the two CPUs of your session run at the same time.

**Predict**: four processes add 1 to the counter, 100,000 times each. Which total do you expect?

In [ ]:
def bump(counter, n):
    for _ in range(n):
        counter.value += 1

counter = multiprocessing.Value("i", 0)

start = time.perf_counter()
workers = [multiprocessing.Process(target=bump, args=(counter, 100_000)) for _ in range(4)]
for p in workers:
    p.start()
for p in workers:
    p.join()

print(f"Counter: {counter.value:,} of 400,000, in {time.perf_counter() - start:.2f} s")

`counter.get_lock()` is the lock that comes with the shared value.

In [ ]:
def safe_bump(counter, n):
    for _ in range(n):
        with counter.get_lock():
            counter.value += 1

counter = multiprocessing.Value("i", 0)

start = time.perf_counter()
workers = [multiprocessing.Process(target=safe_bump, args=(counter, 100_000)) for _ in range(4)]
for p in workers:
    p.start()
for p in workers:
    p.join()

print(f"Counter: {counter.value:,} of 400,000, in {time.perf_counter() - start:.2f} s")

### Share nothing

Most of the time we do not need a shared counter at all. Each worker counts on its own and returns its count, and the
main process adds up the four results.

In [ ]:
def count(n):
    c = 0
    for _ in range(n):
        c += 1
    return c

start = time.perf_counter()
with ProcessPoolExecutor(4) as ex:
    total = sum(ex.map(count, [100_000] * 4))

print(f"Total: {total:,} of 400,000, in {time.perf_counter() - start:.2f} s")

Stop here and answer the questions of **Exercise 3** on the slides.

## Part 4. Homework

### How does the time change with the number of workers?

The next cell writes `speedup.py`. It runs 16 tasks one after the other, then in a process pool with 1, 2, 4 and 8
workers. The **speedup** is the time of one worker divided by the time of `n` workers. The **efficiency** is the speedup
divided by `n`. We come back to both in Session 23.

In [ ]:
%%writefile speedup.py
import os
import time
from concurrent.futures import ProcessPoolExecutor

TASKS = [10_000_000] * 16


def work(n):
    total = 0
    for i in range(n):
        total += i
    return total


def in_a_pool(workers):
    with ProcessPoolExecutor(workers) as ex:
        return list(ex.map(work, TASKS))


def best_of_3(function, *args):
    times = []
    for _ in range(3):
        start = time.perf_counter()
        function(*args)
        times.append(time.perf_counter() - start)
    return min(times)


if __name__ == "__main__":
    print(f"Node {os.uname().nodename}, {len(os.sched_getaffinity(0))} CPUs")
    print("workers   time (s)   speedup   efficiency")
    one = best_of_3(in_a_pool, 1)
    for workers in [1, 2, 4, 8]:
        elapsed = best_of_3(in_a_pool, workers)
        print(f"{workers:7d} {elapsed:10.2f} {one / elapsed:9.2f} {one / elapsed / workers:12.2f}")

Run it with 8 CPUs and then with 16 CPUs. In which rows does the speedup stop growing, and why? Write down your
two tables: we use them in Session 23.

In [ ]:
!sbatch -p cpu -c 8 --mem=2G -t 2 -o speedup-8cpus.out --wait --wrap "python speedup.py"
!cat speedup-8cpus.out

In [ ]:
!sbatch -p cpu -c 16 --mem=2G -t 2 -o speedup-16cpus.out --wait --wrap "python speedup.py"
!cat speedup-16cpus.out

### Many tiny tasks

A process pool sends every item to a worker and every result back. That costs little compared with a task of 0.3 s, and
a lot compared with `abs`, which takes less than a millionth of a second. `chunksize` sends the items in groups.

**Predict**: rank the three times below, from fastest to slowest. The middle one takes a few seconds.

In [ ]:
start = time.perf_counter()
results = list(map(abs, range(100_000)))
print(f"Plain loop: {time.perf_counter() - start:.4f} s")

start = time.perf_counter()
with ProcessPoolExecutor(2) as ex:
    results = list(ex.map(abs, range(100_000)))
print(f"2 processes: {time.perf_counter() - start:.4f} s")

start = time.perf_counter()
with ProcessPoolExecutor(2) as ex:
    results = list(ex.map(abs, range(100_000), chunksize=1000))
print(f"2 processes, chunksize=1000: {time.perf_counter() - start:.4f} s")

### Your project

Choose one slow step of your project. Is it computing or waiting? Which kind of worker fits it, and how many CPUs
would you ask for? Bring your answer to Session 23.